In [1]:
import numpy as np

class GridWorld:
    def __init__(self):
        self.grid = np.array([
            ['S', '.', '.', '.'],
            ['.', 'X', '.', '.'],
            ['.', '.', '.', 'G']
        ])
        self.start = (0, 0)
        self.goal = (2, 3)
        self.pos = self.start
        self.actions = ['up', 'down', 'left', 'right']
    
    def reset(self):
        print("\n--- Resetting environment ---")
        self.pos = self.start
        print(f"Starting position: {self.pos}")
        return self.pos
    
    def step(self, action):
        i, j = self.pos
        print(f"\nCurrent position: {self.pos} | Action: {action}")

        # Move according to action
        if action == 'up': i -= 1
        elif action == 'down': i += 1
        elif action == 'left': j -= 1
        elif action == 'right': j += 1
        
        # Boundary / obstacle checks
        if i < 0 or i >= 3 or j < 0 or j >= 4 or self.grid[i][j] == 'X':
            reward = -1
            next_state = self.pos
            print(f"Hit wall or obstacle at ({i},{j})! Staying at {self.pos}")
        else:
            self.pos = (i, j)
            reward = 1 if self.pos == self.goal else 0
            print(f"Moved to {self.pos}, reward = {reward}")
        
        done = self.pos == self.goal
        if done:
            print("🎯 Goal reached!")
        return self.pos, reward, done


In [2]:
import random

class QLearningAgent:
    def __init__(self, env, alpha=0.1, gamma=0.9, epsilon=0.1):
        self.q_table = {}
        self.alpha = alpha  # learning rate
        self.gamma = gamma  # discount factor
        self.epsilon = epsilon  # exploration rate
        self.env = env
    
    def get_q(self, state, action):
        return self.q_table.get((state, action), 0.0)
    
    def choose_action(self, state):
        # Exploration vs Exploitation
        if random.random() < self.epsilon:
            action = random.choice(self.env.actions)
            print(f"Exploring! Randomly picked action: {action}")
        else:
            q_values = [self.get_q(state, a) for a in self.env.actions]
            max_q = max(q_values)
            action = self.env.actions[q_values.index(max_q)]
            print(f"Exploiting! Best action for {state} is '{action}' with Q={max_q:.2f}")
        return action
    
    def learn(self, state, action, reward, next_state, done):
        old_q = self.get_q(state, action)
        next_qs = [self.get_q(next_state, a) for a in self.env.actions]
        max_next_q = max(next_qs) if not done else 0
        
        new_q = old_q + self.alpha * (reward + self.gamma * max_next_q - old_q)
        self.q_table[(state, action)] = new_q

        print(f"Updated Q-value for {state}, {action}: {old_q:.2f} → {new_q:.2f}")


In [3]:
env = GridWorld()
agent = QLearningAgent(env)

for episode in range(3):  # run just a few to see debug info clearly
    print(f"\n==================== Episode {episode + 1} ====================")
    state = env.reset()
    done = False
    step_count = 0
    
    while not done and step_count < 20:  # limit to avoid infinite loops
        print(f"\nStep {step_count + 1}")
        action = agent.choose_action(state)
        next_state, reward, done = env.step(action)
        agent.learn(state, action, reward, next_state, done)
        
        print(f"Transition: {state} --{action}/{reward}--> {next_state}")
        
        state = next_state
        step_count += 1

    print(f"Episode {episode + 1} finished after {step_count} steps.")



==================== Episode 1 ====================

--- Resetting environment ---
Starting position: (0, 0)

Step 1
Exploiting! Best action for (0, 0) is 'up' with Q=0.00

Current position: (0, 0) | Action: up
Hit wall or obstacle at (-1,0)! Staying at (0, 0)
Updated Q-value for (0, 0), up: 0.00 → -0.10
Transition: (0, 0) --up/-1--> (0, 0)

Step 2
Exploiting! Best action for (0, 0) is 'down' with Q=0.00

Current position: (0, 0) | Action: down
Moved to (1, 0), reward = 0
Updated Q-value for (0, 0), down: 0.00 → 0.00
Transition: (0, 0) --down/0--> (1, 0)

Step 3
Exploiting! Best action for (1, 0) is 'up' with Q=0.00

Current position: (1, 0) | Action: up
Moved to (0, 0), reward = 0
Updated Q-value for (1, 0), up: 0.00 → 0.00
Transition: (1, 0) --up/0--> (0, 0)

Step 4
Exploiting! Best action for (0, 0) is 'down' with Q=0.00

Current position: (0, 0) | Action: down
Moved to (1, 0), reward = 0
Updated Q-value for (0, 0), down: 0.00 → 0.00
Transition: (0, 0) --down/0--> (1, 0)

Step 5
Ex